# Doohan sessions to flat and hierarchical LMDPs

This notebook loads a configurable set of sessions from one Doohan maze, reduces every navigation trial to entered maze towers, scores the resulting independent trajectories under flat and hierarchical goal-conditioned LMDPs, and fits the hierarchy parameters by maximum likelihood.

## 1. Setup paths, select sessions, and build the shared maze

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Run this notebook from the project or notebook directory")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from andrew_mlmdp import (  # noqa: E402
    DoohanDataset,
    Environment,
    NMFConfig,
    SubgoalBasis,
    discover_subgoals,
    fittable_parameters,
    score_flat_dataset,
    score_hierarchy_dataset,
    soft_parameters,
)
from andrew_mlmdp import (  # noqa: E402
    plotting as viz,
)

GRIDMAZE_DATA = (
    PROJECT_ROOT / "external" / "GridMaze-mFC-ephys-DATA" / "data"
)

SUBJECT_IDS = ["m2"]
MAZE_NAME = "maze_1"
START_DATE = "2022-06-28"
END_DATE = "2022-07-05"

movement_dataset = DoohanDataset.from_data_root(
    GRIDMAZE_DATA,
    subject_ids=SUBJECT_IDS,
    start_date=START_DATE,
    end_date=END_DATE,
    maze_name=MAZE_NAME,
)
sessions = movement_dataset.sessions
labeled_maze = movement_dataset.definition
maze = labeled_maze.maze
environment = Environment(maze)

print(f"sessions: {len(sessions)}")
print(f"maze: {movement_dataset.maze_name}")
print(f"tower grid shape: {maze.shape}")
print(f"physical states: {len(maze.free_cells)}")

## 2. Inspect the assembled navigation trials

The Doohan movement dataset owns session discovery and trial extraction. It removes missing positions and bridge labels, collapses consecutive towers, converts labels to coordinates, truncates each trajectory at its first goal entry, and retains malformed sessions or trials as explicit exclusions.

In [ ]:
movement_trials = list(movement_dataset.trials)
exclusion_rows = [
    {
        "session": exclusion.session_id,
        "trial": exclusion.trial_id,
        "goal": exclusion.goal_label,
        "transitions": 0,
        "flat_log_likelihood": float("nan"),
        "hierarchical_log_likelihood": float("nan"),
        "status": "excluded",
        "exclusion_reason": f"data: {exclusion.reason}",
    }
    for exclusion in movement_dataset.exclusions
]

print(f"valid trials: {len(movement_trials)}")
print(f"data exclusions: {len(exclusion_rows)}")

## 3. Inspect the shared maze and one example trial

In [ ]:
tower_labels = dict(labeled_maze.label_by_coordinate)
figure = viz.plot_maze(
    maze,
    labels=tower_labels,
    title=f"{MAZE_NAME}: {len(sessions)} selected sessions",
)
if movement_trials:
    example_trial = movement_trials[0]
    example_start = example_trial.trajectory[0]
    figure.add_trace(go.Scatter(
        x=[example_start[1], example_trial.goal[1]],
        y=[example_start[0], example_trial.goal[0]],
        mode="markers",
        marker={
            "symbol": ["circle", "star"],
            "size": [12, 17],
            "color": ["#4c956c", "#d1495b"],
        },
        text=[
            f"start ({labeled_maze.label_for(example_start)})",
            f"goal ({labeled_maze.label_for(example_trial.goal)})",
        ],
        hovertemplate="%{text}<extra></extra>",
        name="example trial",
    ))
figure.show()

## 4. Build one flat environment and one hierarchy template

The distributed basis is discovered once for the maze, not once per session or trial. Goal-conditioned flat solutions and hierarchy tasks are then created lazily and cached by the dataset scorers.

In [ ]:
discovery_rank = 8
discovery_control_cost = 3.0
soft_discovery = discover_subgoals(
    environment,
    ranks=(discovery_rank,),
    parameters=NMFConfig(control_cost=discovery_control_cost),
    seed=0,
).result(discovery_rank)
viz.plot_subtasks(soft_discovery).show()

soft_basis = SubgoalBasis.from_profiles(
    maze,
    soft_discovery.profiles,
    core_threshold=0.8,
)
hierarchy_template = environment.hierarchy(
    soft_basis,
    parameters=soft_parameters(
        discovery_rank,
        upper_control_cost=1.8,
    ),
)


In [ ]:
trial = movement_trials[0]

start = trial.trajectory[0]
goal = trial.goal

task = hierarchy_template.task(goal)

simulated = task.rollout(
    start,
    goal_learning="online",
    max_steps=500,
)

print(simulated.status)
print(simulated.reached_goal)
print(simulated.trajectory)

player = viz.explore_rollout(
    hierarchy_template,
    start,
    goal,
    max_steps=500,
)

display(player.controls)
player.figure.show()

## 5. Generate one report per model

Each report represents one model result on the selected dataset. The flat and hierarchical reports remain independent; this notebook displays only the hierarchical dataset summary.

In [ ]:
flat_report = movement_dataset.report(
    score_flat_dataset(
        environment,
        movement_trials,
    )
)
hierarchical_report = movement_dataset.report(
    score_hierarchy_dataset(
        hierarchy_template,
        movement_trials,
    )
)

display(hierarchical_report.summary_dataframe())
display(hierarchical_report.trial_dataframe())
display(hierarchical_report.session_dataframe())


## 6. Fit the hierarchy parameters

The differentiable fitter stops if any fitted trajectory has a nonfinite likelihood. The code below therefore fits the subset that is finite under the initial `hierarchy_template` and reports how many trials were omitted. It optimizes every parameter active for this template and returns the best finite Adam state without mutating the template.

In [ ]:
# Filter out trials with nonfinite initial likelihoods
finite_trial_keys = {
    (score.session_id, score.trial_id)
    for score in hierarchical_report.result.trial_likelihoods
    if math.isfinite(score.log_likelihood)
}
fitting_trials = tuple(
    trial
    for trial in movement_trials
    if (trial.session_id, trial.trial_id) in finite_trial_keys
)
omitted_trial_count = len(movement_trials) - len(fitting_trials)
print(f"fitting trials: {len(fitting_trials)}")
print(f"omitted nonfinite initial trials: {omitted_trial_count}")
if not fitting_trials:
    raise RuntimeError("No trials have a finite initial hierarchy likelihood")

In [ ]:
FIT_LEARNING_RATE = 5e-2
FIT_MAX_STEPS = 250
FIT_RELATIVE_TOLERANCE = 1e-8
FIT_PATIENCE = 20
FIT_LR_DECAY_FACTOR = 0.3
FIT_LR_DECAY_PATIENCE = 7
FIT_MINIMUM_LEARNING_RATE = 1e-5

progress_display = display(
    "Waiting for first Adam evaluation...",
    display_id=True,
)

def show_fit_progress(evaluation):
    completed = min(evaluation.updates, FIT_MAX_STEPS)
    fraction = completed / max(1, FIT_MAX_STEPS)
    filled = round(30 * fraction)
    bar = "█" * filled + "·" * (30 - filled)
    best_loss = evaluation.best_loss
    best_loss_text = (
        "n/a" if best_loss is None else f"{best_loss:.6f}"
    )
    progress_display.update(
        f"[{bar}] {completed}/{FIT_MAX_STEPS} Adam updates | "
        f"loss={evaluation.loss:.6f} | best={best_loss_text} | "
        f"lr={evaluation.lr:.3e} | "
        f"gradient norm={evaluation.gradient_norm:.3e}"
    )

fit_names = fittable_parameters(hierarchy_template)
fit_result = hierarchy_template.fit(
    fitting_trials,
    names=fit_names,
    lr=FIT_LEARNING_RATE,
    max_steps=FIT_MAX_STEPS,
    tolerance=FIT_RELATIVE_TOLERANCE,
    patience=FIT_PATIENCE,
    lr_decay=FIT_LR_DECAY_FACTOR,
    lr_patience=FIT_LR_DECAY_PATIENCE,
    min_lr=FIT_MINIMUM_LEARNING_RATE,
    callback=show_fit_progress,
)
if fit_result.best_values is None:
    raise RuntimeError(
        "Parameter fitting ended before finding a finite state: "
        f"{fit_result.reason}"
    )

best_parameter_values = fit_result.best_values.as_floats()
optimal_parameter_values = {
    name: best_parameter_values[name]
    for name in fit_names
}
parameter_table = pd.DataFrame(
    {
        "initial_value": pd.Series(
            {
                name: fit_result.initial_values.as_floats()[name]
                for name in fit_names
            }
        ),
        "optimal_value": pd.Series(optimal_parameter_values),
    }
).rename_axis("parameter")
finite_evaluations = [
    evaluation
    for evaluation in fit_result.history
    if math.isfinite(evaluation.loss)
]
best_evaluation = min(
    finite_evaluations,
    key=lambda evaluation: evaluation.loss,
)
fit_summary = pd.DataFrame(
    [
        {
            "fitted_trials": len(fitting_trials),
            "omitted_initial_nonfinite": omitted_trial_count,
            "optimizer_updates": fit_result.updates,
            "reason": fit_result.reason,
            "converged": fit_result.converged,
            "initial_total_log_likelihood": (
                fit_result.history[0].log_likelihood
            ),
            "best_total_log_likelihood": (
                best_evaluation.log_likelihood
            ),
            "best_gradient_norm": best_evaluation.gradient_norm,
        }
    ]
)
display(fit_summary)
display(parameter_table)
optimal_parameter_values

## 7. Visualize one flat solution and observed trajectory

In [ ]:
example_trial = movement_trials[0] if movement_trials else None
if example_trial is not None:
    example_solution = environment.solve(example_trial.goal)
    goal_state = maze.state_index(example_trial.goal)
    relative_desirability = (
        example_solution.desirability
        / example_solution.desirability[goal_state]
    )
    plot_values = np.where(
        relative_desirability > 0.0,
        np.log10(relative_desirability),
        np.nan,
    )
    figure = viz.plot_maze(maze, show_grid=False, title=None)
    figure.add_trace(go.Scatter(
        x=[coordinate[1] for coordinate in maze.free_cells],
        y=[coordinate[0] for coordinate in maze.free_cells],
        mode="markers",
        marker={
            "size": 18,
            "color": plot_values,
            "colorscale": "Viridis",
            "colorbar": {"title": "log10 relative desirability"},
            "line": {"color": "white", "width": 1},
        },
        customdata=relative_desirability,
        hovertemplate="relative desirability: %{customdata:.4g}<extra></extra>",
        name="desirability",
    ))
    viz.plot_trajectory_overlay(
        maze,
        example_trial.trajectory,
        goal=example_trial.goal,
        fig=figure,
    )
    figure.update_layout(title=(
        "Flat-LMDP and behavior for "
        f"{example_trial.session_id}, trial {example_trial.trial_id}"
    ))
    figure.show()